# 05 — Shor Code, CSS Codes, and the Steane Code

## Read before you begin

- Arthur Pesah, [*Stabilizer Formalism I*](https://arthurpesah.me/blog/2023-01-31-stabilizer-formalism-1/).
- Sections to read: **Our first truly quantum code: Shor's code**, **Stabilizer formalism: first definitions**, **Quantum codes from classical codes: the CSS construction**, **Steane code, the quantum version of the Hamming code**.
- Concepts to extract: how Shor's code nests a bit-flip code inside a phase-flip code; the general definition of a stabilizer group and code space (beyond the two-generator repetition-code case from notebook 03); the CSS recipe for turning classical parity checks into commuting quantum stabilizers; why the Hamming code specifically can generate *both* the $X$-type and $Z$-type stabilizers of a quantum code.

This notebook does not re-derive what a stabilizer or a syndrome is — notebook 03 already covers that. It builds directly on those definitions at a larger scale, and — unlike an earlier version of this notebook — it builds the codes as *real circuits* using [Loom](https://loom-docs.entropicalabs.com/) (Entropica Labs' QEC circuit-construction tool, already used in notebook 10) and [Stim](https://github.com/quantumlib/Stim), instead of hand-built NumPy matrices. Loom ships dedicated `ShorCode` and `SteaneCode` block factories — real, verified circuit constructions, not something this notebook has to derive from scratch.

## Learning objectives

By the end of this notebook you should be able to:

1. build Shor's $[[9,1,3]]$ code as an actual Stim circuit via Loom's `ShorCode` factory, and read off its 8 real stabilizer generators;
2. detect a bit-flip and a phase-flip error by sampling real circuit detectors, and explain *degeneracy* — why several different physical errors give the identical detector outcome — directly from that sampled data;
3. verify computationally that the Shor code is $[[9,1,3]]$: 1 logical qubit from 9 physical qubits, distance 3;
4. state the general CSS construction and explain why it needs a classical code whose parity checks are pairwise orthogonal;
5. build the Steane code as a real circuit via Loom's `SteaneCode` factory, verify its $X$/$Z$ stabilizers commute using Stim's own `PauliString.commutes()`, and read off single-error syndromes from sampled detector data.

**Expected outputs:** a real, Loom-built Stim circuit for each code; an empirically-sampled single-error syndrome table for all 9 (Shor) and 7 (Steane) data qubits, showing degeneracy and unique error identification directly from circuit behavior; and a computational distance-3 proof.

**Assumes:** notebook 03 (stabilizers, syndromes, eigenvalues), notebook 04 (the $[7,4,3]$ Hamming code's CSS-orthogonality property), and notebook 10 (Loom's `Eka`/`Lattice`/operations pattern, `EkaToStimConverter`).

## 1. Building the Shor code as a real circuit

Loom's `ShorCode.create(lattice)` builds exactly the $[[9,1,3]]$ code: 9 data qubits in 3 blocks of 3, 2 weight-6 $X$-type stabilizers (one spanning blocks A+B, one spanning B+C — exactly notebook 03's "which blocks does this check touch" idea, now at block scale) and 6 weight-2 $Z$-type stabilizers (one pair per block, literally notebook 02/03's repetition-code check, repeated 3 times).

In [1]:
from loom.eka import Eka, Lattice
from loom.eka.operations import ResetAllDataQubits, MeasureBlockSyndromes, MeasureLogicalZ
from loom.interpreter import interpret_eka
from loom.executor import EkaToStimConverter
from loom_shor_code.code_factory import ShorCode
import stim

shor_lattice = Lattice.linear((9,))
shor_code = ShorCode.create(lattice=shor_lattice, unique_label="shor9")

print(f"{len(shor_code.stabilizers)} stabilizer generators:")
for s in shor_code.stabilizers:
    print(f"  {s.pauli:<6} on data qubits {s.data_qubits}")

print(f"\n{shor_code.n_logical_qubits} logical qubit(s).")
print("Loom's own logical operators (not necessarily minimum weight):")
for s in shor_code.logical_x_operators:
    print(f"  logical X: {s.pauli} on {s.data_qubits}")
for s in shor_code.logical_z_operators:
    print(f"  logical Z: {s.pauli} on {s.data_qubits}")

8 stabilizer generators:
  XXXXXX on data qubits ((0, 0), (1, 0), (2, 0), (3, 0), (4, 0), (5, 0))
  XXXXXX on data qubits ((3, 0), (4, 0), (5, 0), (6, 0), (7, 0), (8, 0))
  ZZ     on data qubits ((0, 0), (1, 0))
  ZZ     on data qubits ((1, 0), (2, 0))
  ZZ     on data qubits ((3, 0), (4, 0))
  ZZ     on data qubits ((4, 0), (5, 0))
  ZZ     on data qubits ((6, 0), (7, 0))
  ZZ     on data qubits ((7, 0), (8, 0))

1 logical qubit(s).
Loom's own logical operators (not necessarily minimum weight):
  logical X: XXXXXXXXX on ((0, 0), (1, 0), (2, 0), (3, 0), (4, 0), (5, 0), (6, 0), (7, 0), (8, 0))
  logical Z: ZZZZZZZZZ on ((0, 0), (1, 0), (2, 0), (3, 0), (4, 0), (5, 0), (6, 0), (7, 0), (8, 0))


Loom reports weight-9 logical operators (all-$X$ and all-$Z$) — a valid but *not* minimum-weight choice. Notebook 06 will show precisely why a lighter representative (weight 3) is logically equivalent to this one; for now, just note that "a" logical operator and "the lightest" logical operator can differ, which is exactly why code *distance* (Section 3) needs its own definition rather than reading a weight off whichever representative a tool happens to report.

### From Loom's description to a Stim circuit

An `Eka` bundles a lattice, one or more code blocks, and a sequence of operations (reset, measure syndromes, measure a logical operator) — exactly the pattern notebook 10 used for the repetition code. `EkaToStimConverter` compiles it to an ordinary Stim circuit.

In [2]:
shor_eka = Eka(
    lattice=shor_lattice,
    blocks=(shor_code,),
    operations=(
        ResetAllDataQubits(input_block_name="shor9", state="0"),
        MeasureBlockSyndromes(input_block_name="shor9", n_cycles=2),
        MeasureLogicalZ(input_block_name="shor9", logical_qubit=0),
    ),
)
shor_circuit, shor_qubit_channels, shor_classical_channels = EkaToStimConverter().convert(interpret_eka(shor_eka))

print(f"{shor_circuit.num_qubits} physical qubits, {shor_circuit.num_detectors} detectors, {shor_circuit.num_observables} observable(s).")
print(shor_circuit)

17 physical qubits, 20 detectors, 1 observable(s).
QUBIT_COORDS(-0.5, 0.5) 0
QUBIT_COORDS(0, 0) 1
QUBIT_COORDS(0.5, 0.5) 2
QUBIT_COORDS(1, 0) 3
QUBIT_COORDS(1.5, 0.5) 4
QUBIT_COORDS(2, 0) 5
QUBIT_COORDS(2.5, 0.5) 6
QUBIT_COORDS(3, 0) 7
QUBIT_COORDS(3.5, 0.5) 8
QUBIT_COORDS(4, 0) 9
QUBIT_COORDS(4.5, 0.5) 10
QUBIT_COORDS(5, 0) 11
QUBIT_COORDS(5.5, 0.5) 12
QUBIT_COORDS(6, 0) 13
QUBIT_COORDS(6.5, 0.5) 14
QUBIT_COORDS(7, 0) 15
QUBIT_COORDS(7.5, 0.5) 16
R 0 2 4 6 8 10 12 14 16 1 3 5 7 9 11 13 15
H 1 3 5 7 9 11 13 15
CX 1 0 3 6 1 2 3 8 1 4 3 10 1 6 3 12 1 8 3 14 1 10 3 16
CZ 5 0 7 2 9 6 11 8 13 12 15 14 5 2 7 4 9 8 11 10 13 14 15 16
H 1 3 5 7 9 11 13 15
M 1 3 5 7 9 11 13 15
R 1 3 5 7 9 11 13 15
H 1 3 5 7 9 11 13 15
CX 1 0 3 6 1 2 3 8 1 4 3 10 1 6 3 12 1 8 3 14 1 10 3 16
CZ 5 0 7 2 9 6 11 8 13 12 15 14 5 2 7 4 9 8 11 10 13 14 15 16
H 1 3 5 7 9 11 13 15
M 1 3 5 7 9 11 13 15 0 2 4 6 8 10 12 14 16
DETECTOR(2, 0, 13) rec[-23]
DETECTOR(3, 0, 13) rec[-22]
DETECTOR(4, 0, 13) rec[-21]
DETECTOR(5, 0, 1

Reading the circuit: `R` resets every qubit to $|0\rangle$; `H`/`CX`/`CZ` prepare the code state and copy each stabilizer's parity onto its own ancilla (the same H-sandwich pattern notebook 09's surface-code ancillas used); the first `M` measures the 8 ancillas, `R` resets them for a second round, and the final `M` measures everything, including the 9 data qubits, for the logical readout. `DETECTOR` lines compare measurements across rounds — exactly the "defect" idea from notebook 09, generalized from a 2D grid to this code's block structure.

## 2. Identifying data vs. ancilla qubits, and injecting a real error

As in notebook 09, we identify roles directly from the circuit rather than assuming an index convention: ancillas are reset more than once (once per round); data qubits are reset only at the very start.

In [3]:
def data_and_ancilla_qubits(circuit):
    reset_instructions = [instr for instr in circuit if instr.name == "R"]
    initial_reset = {t.value for t in reset_instructions[0].targets_copy()}
    ancillas = set()
    for instr in reset_instructions[1:]:
        ancillas |= {t.value for t in instr.targets_copy()}
    return sorted(initial_reset - ancillas), sorted(ancillas)


shor_data_qubits, shor_ancilla_qubits = data_and_ancilla_qubits(shor_circuit)
print(f"{len(shor_data_qubits)} data qubits: {shor_data_qubits}")
print(f"{len(shor_ancilla_qubits)} ancilla qubits: {shor_ancilla_qubits}")

9 data qubits: [0, 2, 4, 6, 8, 10, 12, 14, 16]
8 ancilla qubits: [1, 3, 5, 7, 9, 11, 13, 15]


To inject one clean, well-defined error, we use `X_ERROR(1.0)`/`Z_ERROR(1.0)` (real noise instructions, not a bare gate — notebook 09 found the hard way that a bare gate is absorbed into the circuit's "expected" trajectory and a `DETECTOR` never flags it) inserted *between* the two syndrome-measurement rounds, so the first round sees the clean value and the second round sees the corrupted one.

In [4]:
def inject_error_between_rounds(circuit, qubit, kind):
    first_m_index = next(i for i, instr in enumerate(circuit) if instr.name == "M")
    corrupted = circuit.copy()
    corrupted.insert(first_m_index + 1, stim.CircuitInstruction(f"{kind}_ERROR", [qubit], [1.0]))
    return corrupted


def fired_detectors(circuit, qubit, kind):
    corrupted = inject_error_between_rounds(circuit, qubit, kind)
    outcome = corrupted.compile_detector_sampler().sample(shots=1)[0]
    return tuple(i for i, bit in enumerate(outcome) if bit)


print("Example: X error on the first data qubit fires detectors", fired_detectors(shor_circuit, shor_data_qubits[0], "X"))

Example: X error on the first data qubit fires detectors (8,)


### Prediction 1

Data qubits 0, 1, 2 (in Loom's block order) are block A; 3, 4, 5 are block B; 6, 7, 8 are block C. **Before running the next cell:** an $X$ error should only ever affect *within-block* $Z$-type checks (same reasoning as notebook 03: $X$ always commutes with $X$, so the 2 weight-6 $X$-type stabilizers never react). A $Z$ error should only affect the $X$-type checks that include its block. Based on Section 1's stabilizer list, which blocks should share an $X$-type check, and which should each have their own?

In [5]:
print(f"{'data idx':>8} {'X error fires':>16} {'Z error fires':>16}")
for i, q in enumerate(shor_data_qubits):
    print(f"{i:>8} {str(fired_detectors(shor_circuit, q, 'X')):>16} {str(fired_detectors(shor_circuit, q, 'Z')):>16}")

data idx    X error fires    Z error fires
       0             (8,)             (6,)
       1           (8, 9)             (6,)
       2             (9,)             (6,)
       3            (10,)           (6, 7)
       4         (10, 11)           (6, 7)
       5            (11,)           (6, 7)
       6            (12,)             (7,)
       7         (12, 13)             (7,)
       8            (13,)             (7,)


Read straight off the sampled data: $X$ errors on qubits within the same block fire overlapping $Z$-check detectors (edge qubits of a block fire one, the middle qubit fires both — exactly notebook 02/03's within-block repetition-code pattern). $Z$ errors are the striking part: **every qubit in block A fires the identical single detector**, every qubit in block B fires the identical pair, every qubit in block C fires the identical single detector — this *is* degeneracy, read directly from sampled circuit data rather than compared state vectors: a decoder watching only these detectors could never tell qubit 0 from qubit 1 apart, and does not need to — correcting "block A" (any fixed representative qubit) fixes the error either way.

## 3. Proving Shor's code is $[[9,1,3]]$

**$k=1$:** already reported directly by Loom (`n_logical_qubits = 1`) — no need to independently derive it here, since it's the code factory's job to guarantee that, not this notebook's.

**$d=3$:** this is a genuine mathematical claim about *every* possible error, not something a single sampled circuit run can demonstrate — so, as in notebook 07, we check it by exhaustive search. The one piece that changes is *how* we test commutation: `stim.PauliString.commutes()` is a real Stim primitive that does exactly what notebook 07's hand-rolled `symplectic_product` function computed by hand.

In [6]:
def stabilizer_full_string(stabilizer, n=9):
    chars = ["I"] * n
    for pauli_char, qubit_coord in zip(stabilizer.pauli, stabilizer.data_qubits):
        chars[shor_code.data_qubits.index(qubit_coord)] = pauli_char
    return "".join(chars)


shor_generator_strings = [stabilizer_full_string(s) for s in shor_code.stabilizers]
shor_generators = [stim.PauliString(s) for s in shor_generator_strings]
print(shor_generator_strings)

# Sanity check: Stim's own commutation test agrees with which detectors fired in Section 2.
x0 = stim.PauliString("X" + "I" * 8)
z4 = stim.PauliString("I" * 4 + "Z" + "I" * 4)
print("X0 commutes with all 8 generators:", [x0.commutes(g) for g in shor_generators])

['XXXXXXIII', 'IIIXXXXXX', 'ZZIIIIIII', 'IZZIIIIII', 'IIIZZIIII', 'IIIIZZIII', 'IIIIIIZZI', 'IIIIIIIZZ']
X0 commutes with all 8 generators: [True, True, False, True, True, True, True, True]


In [7]:
from itertools import combinations, product as iproduct


def all_weight_le(max_weight, n=9):
    strings = []
    for w in range(1, max_weight + 1):
        for positions in combinations(range(n), w):
            for letters in iproduct("XYZ", repeat=w):
                chars = ["I"] * n
                for pos, letter in zip(positions, letters):
                    chars[pos] = letter
                strings.append("".join(chars))
    return strings


def logical_action_is_trivial(pauli_string, logical_x, logical_z):
    """A commuting operator is trivial (a stabilizer) iff it also commutes with both
    logical operators -- an operator that anticommutes with a logical operator is
    itself a nontrivial logical operator (it changes what a logical measurement reads)."""
    p = stim.PauliString(pauli_string)
    return p.commutes(logical_x) and p.commutes(logical_z)


logical_x_full = stim.PauliString(stabilizer_full_string(shor_code.logical_x_operators[0]))
logical_z_full = stim.PauliString(stabilizer_full_string(shor_code.logical_z_operators[0]))

low_weight_candidates = all_weight_le(2)
nontrivial_low_weight = []
for candidate in low_weight_candidates:
    p = stim.PauliString(candidate)
    if all(p.commutes(g) for g in shor_generators):
        if not logical_action_is_trivial(candidate, logical_x_full, logical_z_full):
            nontrivial_low_weight.append(candidate)

print(f"Checked {len(low_weight_candidates)} weight-1 and weight-2 Pauli strings.")
print("Nontrivial logical operators found at weight <= 2:", nontrivial_low_weight)
print("Distance is therefore exactly 3:", len(nontrivial_low_weight) == 0)

Checked 351 weight-1 and weight-2 Pauli strings.
Nontrivial logical operators found at weight <= 2: []
Distance is therefore exactly 3: True


No weight-1 or weight-2 Pauli string both evades every stabilizer *and* acts as a genuine logical operator, while Section 1 already exhibited weight-9 logical operators (and notebook 06 will exhibit weight-3 ones equivalent to them). **The Shor code is exactly $[[9,1,3]]$.**

## 4. The CSS construction: quantum codes from classical codes

The **CSS construction** (Calderbank–Shor–Steane) is a general recipe: take a classical linear code's parity-check matrix $H$ (rows = classical parity checks, as in notebook 04) and turn *each row* into two quantum stabilizers — one built entirely from $X$s on that row's support, one entirely from $Z$s.

Row $h_i$ as an $X$-type stabilizer and row $h_j$ as a $Z$-type stabilizer commute exactly when they overlap in an even number of positions, i.e. when $h_i \cdot h_j = 0 \pmod 2$. So the whole construction works — *every* $X$-type generator commutes with *every* $Z$-type generator — precisely when

$$H H^T \equiv 0 \pmod 2.$$

This is a property of a *classical* matrix, not a quantum experiment, so it stays plain Python — there is nothing for Stim or Loom to simulate here.

In [8]:
def bits_of(j, width):
    return tuple((j >> shift) & 1 for shift in reversed(range(width)))


# Same construction as notebook 04: H's columns are the binary representations of 1..7.
H_columns = [bits_of(j, width=3) for j in range(1, 8)]
H = tuple(tuple(col[row] for col in H_columns) for row in range(3))

import numpy as np

H_np = np.array(H)
HHt_mod2 = (H_np @ H_np.T) % 2
print("H =")
for row in H:
    print(row)
print("H H^T mod 2 =")
print(HHt_mod2)
print("All zero (checks are pairwise orthogonal):", np.all(HHt_mod2 == 0))

H =
(0, 0, 0, 1, 1, 1, 1)
(0, 1, 1, 0, 0, 1, 1)
(1, 0, 1, 0, 1, 0, 1)
H H^T mod 2 =
[[0 0 0]
 [0 0 0]
 [0 0 0]]
All zero (checks are pairwise orthogonal): True


The Hamming code's parity-check matrix satisfies $HH^T \equiv 0 \pmod 2$ — it is **dual-containing**. That single fact is the entire reason a Hamming-derived quantum code can generate both the $X$-type and $Z$-type stabilizers from one classical code — no second classical code is needed. Section 5 builds exactly such a code, via Loom's own $[[7,1,3]]$ block factory rather than by hand-deriving stabilizers from this specific $H$.

## 5. The Steane code, as a real circuit

Loom's `SteaneCode` factory builds a genuine $[[7,1,3]]$ CSS code: 3 $X$-type and 3 $Z$-type weight-4 stabilizers on 7 data qubits (its own qubit layout, on a 2D lattice — not necessarily numbered the same way as notebook 04's $H$, but structurally the same CSS/Hamming-derived code: 6 independent checks, $k = 7-6=1$).

In [9]:
from loom_steane_code.code_factory import SteaneCode

steane_lattice = Lattice.square_2d(lattice_size=(4, 4))
steane_code = SteaneCode.create(lattice=steane_lattice, unique_label="steane7")

print(f"{len(steane_code.stabilizers)} stabilizer generators:")
for s in steane_code.stabilizers:
    print(f"  {s.pauli:<6} on data qubits {s.data_qubits}")
print(f"{steane_code.n_logical_qubits} logical qubit(s); logical X = {steane_code.logical_x_operators[0].pauli} on {steane_code.logical_x_operators[0].data_qubits}")

6 stabilizer generators:
  XXXX   on data qubits ((0, 0, 0), (0, 1, 0), (1, 1, 0), (1, 0, 0))
  XXXX   on data qubits ((1, 0, 0), (1, 1, 0), (2, 1, 0), (2, 0, 0))
  XXXX   on data qubits ((2, 1, 0), (1, 2, 0), (0, 1, 0), (1, 1, 0))
  ZZZZ   on data qubits ((0, 0, 0), (0, 1, 0), (1, 1, 0), (1, 0, 0))
  ZZZZ   on data qubits ((1, 0, 0), (1, 1, 0), (2, 1, 0), (2, 0, 0))
  ZZZZ   on data qubits ((2, 1, 0), (1, 2, 0), (0, 1, 0), (1, 1, 0))
1 logical qubit(s); logical X = XXX on ((0, 0, 0), (1, 0, 0), (2, 0, 0))


### Verification: direct commutation check with Stim

Rather than building $128\times128$ matrices by hand, ask Stim directly whether every $X$-type generator commutes with every $Z$-type one.

In [10]:
def steane_full_string(stabilizer):
    n = len(steane_code.data_qubits)
    chars = ["I"] * n
    for pauli_char, qubit_coord in zip(stabilizer.pauli, stabilizer.data_qubits):
        chars[steane_code.data_qubits.index(qubit_coord)] = pauli_char
    return "".join(chars)


steane_x_strings = [steane_full_string(s) for s in steane_code.stabilizers if "X" in s.pauli]
steane_z_strings = [steane_full_string(s) for s in steane_code.stabilizers if "Z" in s.pauli]

all_commute = True
for sx in steane_x_strings:
    for sz in steane_z_strings:
        commute = stim.PauliString(sx).commutes(stim.PauliString(sz))
        all_commute &= commute
        print(f"[{sx}, {sz}] commute: {commute}")

print("\nEvery X-type generator commutes with every Z-type generator:", all_commute)

[XXXXIII, ZZZZIII] commute: True
[XXXXIII, IIZZIZZ] commute: True
[XXXXIII, IZIZZIZ] commute: True
[IIXXIXX, ZZZZIII] commute: True
[IIXXIXX, IIZZIZZ] commute: True
[IIXXIXX, IZIZZIZ] commute: True
[IXIXXIX, ZZZZIII] commute: True
[IXIXXIX, IIZZIZZ] commute: True
[IXIXXIX, IZIZZIZ] commute: True

Every X-type generator commutes with every Z-type generator: True


### Prediction 2

**Before running the next cell:** build the Steane code's Stim circuit exactly as Section 1 did for Shor's code, then inject single $X$ and $Z$ errors on each of its 7 data qubits. Notebook 04 showed a $[7,4,3]$ Hamming code's single-bit-error syndromes are all distinct and nonzero. Do you expect the same to hold here, even though Loom's qubit layout is not the same as notebook 04's column ordering?

In [11]:
steane_eka = Eka(
    lattice=steane_lattice,
    blocks=(steane_code,),
    operations=(
        ResetAllDataQubits(input_block_name="steane7", state="0"),
        MeasureBlockSyndromes(input_block_name="steane7", n_cycles=2),
        MeasureLogicalZ(input_block_name="steane7", logical_qubit=0),
    ),
)
steane_circuit, _, _ = EkaToStimConverter().convert(interpret_eka(steane_eka))
steane_data_qubits, steane_ancilla_qubits = data_and_ancilla_qubits(steane_circuit)
print(f"{len(steane_data_qubits)} data qubits, {len(steane_ancilla_qubits)} ancillas, {steane_circuit.num_detectors} detectors")

print(f"\n{'data idx':>8} {'X error fires':>20} {'Z error fires':>20}")
for i, q in enumerate(steane_data_qubits):
    print(f"{i:>8} {str(fired_detectors(steane_circuit, q, 'X')):>20} {str(fired_detectors(steane_circuit, q, 'Z')):>20}")

7 data qubits, 6 ancillas, 12 detectors

data idx        X error fires        Z error fires
       0                 (6,)                 (3,)
       1               (6, 8)               (3, 5)
       2               (6, 7)               (3, 4)
       3            (6, 7, 8)            (3, 4, 5)
       4                 (8,)                 (5,)
       5                 (7,)                 (4,)
       6               (7, 8)               (4, 5)


Every one of the 7 data qubits produces a distinct, nonzero fired-detector pattern for each error type — the same "unique syndrome per single-qubit error" property notebook 04 found for the classical Hamming code, now confirmed for a real Steane-code circuit built by an independent tool (Loom), using its own qubit layout rather than notebook 04's specific $H$. The CSS construction guarantees this structurally, regardless of exactly how a particular implementation labels its qubits.

## Exercises

**1.** Section 2 showed block B's three data qubits (indices 3, 4, 5) all give the identical $Z$-error detector pattern. Write `verify_degenerate_correction(circuit, data_qubits, block_indices, fixed_correction_index)` that, for each qubit index in `block_indices`, builds the corrupted circuit (`inject_error_between_rounds` with kind `"Z"`), and confirms `fired_detectors` is identical across all of them — turning Section 2's visual observation into an automated check. Run it on block A (indices 0, 1, 2) and block C (indices 6, 7, 8) as well.

**2.** Section 3's exhaustive search used `stim.PauliString.commutes()` for the *filtering* step but Python's plain iteration for the search itself. Write `count_stabilizer_group_elements(generators)` that returns how many distinct sign-free Pauli strings arise from every possible product of a subset of `shor_generators` (hint: there are $2^8$ subsets — multiply each subset's Pauli strings together with `stim.PauliString` multiplication, `*`, and collect the distinct string representations). Does the count match $2^{|generators|}$, and how does that relate to notebook 06's centralizer/stabilizer-group discussion?

In [12]:
def verify_degenerate_correction(circuit, data_qubits, block_indices, fixed_correction_index):
    pass  # TODO: implement, then remove this stub body


def count_stabilizer_group_elements(generators):
    pass  # TODO: implement, then remove this stub body


# TODO: once implemented, try e.g.:
# print(verify_degenerate_correction(shor_circuit, shor_data_qubits, [0, 1, 2], 0))
# print(verify_degenerate_correction(shor_circuit, shor_data_qubits, [6, 7, 8], 6))
# print(count_stabilizer_group_elements(shor_generators))